# ML-07 — Baseline Action Score and Top-20 Review

The original assignment uses the FlyRank Hugging Face warehouse dataset. Due to repeated access issues with the gated dataset, I completed this notebook using the dataset provided inside the internship repository:

`../../data/raw/content_refresh_anonymized.csv`

This CSV contains the same type of anonymized search performance data required for learning the concepts of data contracts, feature engineering, and data leakage. All queries, feature engineering, and verification steps in this notebook are therefore performed on the repository CSV instead of the Hugging Face warehouse tables.

## My rule

I use a simple refresh-priority rule based on two signals that are available in the current dataset:

1. **Content staleness** — measured using `days_since_last_update`.
2. **CTR weakness relative to search position** — measured using `ctr` and `avg_position`.

The rule gives a higher score to content that has not been updated for a long time and content that receives a weak CTR for its current search position.

### Reason codes

- `STALE_CONTENT` — the page has not been updated for a long time.
- `WEAK_CTR_POSITION` — the page has a relatively weak CTR for its search position.
- `STALE_AND_WEAK_CTR` — both signals indicate a refresh opportunity.
- `NO_STRONG_SIGNAL` — neither signal is strong enough for a high-priority refresh.

### Action labels

- `HIGH_PRIORITY_REFRESH` — strong evidence from both signals.
- `REVIEW_REFRESH` — at least one important refresh signal is present.
- `LOW_PRIORITY_REVIEW` — weak or limited evidence.

The score is only a baseline for ranking pages for human review. It is not treated as a final decision.

In [1]:
# Load the repository CSV and inspect the columns used by the rule

import pandas as pd

data_path = "../../data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns used by the rule:")

display(
    df[
        [
            "content_id",
            "days_since_last_update",
            "ctr",
            "avg_position"
        ]
    ].head()
)

Rows: 30000
Columns used by the rule:


,content_id,days_since_last_update,ctr,avg_position
0,content_304f48230142,20,0.76,10.6
1,content_a1fb4e703a9e,25,0.05,20.3
2,content_9aa793d4d895,20,0.09,36.5
3,content_331d6c4de07b,22,0.49,6.2
4,content_d99b7a2d90ca,14,0.13,44.0


## 2. Build the ranked queue (writes the CSV)

The score is built only from signals that are available in the repository dataset.

The rule gives:

- 2 points when content has not been updated for at least 180 days.
- 1 point when content has not been updated for 90–179 days.
- 1 point when CTR is weak for the page's average search position.

The final score is used only to rank the pages.

The notebook writes the ranked queue to:

`work/outputs/baseline_action_score.csv`

In [2]:
# Prepare the rule inputs

df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"], errors="coerce"
)

df["ctr"] = pd.to_numeric(
    df["ctr"], errors="coerce"
)

df["avg_position"] = pd.to_numeric(
    df["avg_position"], errors="coerce"
)

# Keep rows usable for the rule
df["days_since_last_update"] = df["days_since_last_update"].fillna(0)
df["ctr"] = df["ctr"].fillna(0)
df["avg_position"] = df["avg_position"].fillna(100)


# Staleness points
df["stale_points"] = 0

df.loc[
    df["days_since_last_update"] >= 180,
    "stale_points"
] = 2

df.loc[
    (df["days_since_last_update"] >= 90) &
    (df["days_since_last_update"] < 180),
    "stale_points"
] = 1

# Weak CTR signal
def weak_ctr(row):

    position = row["avg_position"]
    ctr = row["ctr"]

    if position <= 5:
        return ctr < 0.05

    elif position <= 10:
        return ctr < 0.03

    else:
        return ctr < 0.015


df["weak_ctr"] = df.apply(weak_ctr, axis=1)

df["ctr_points"] = df["weak_ctr"].astype(int)


# Final score

df["score"] = (
    df["stale_points"] +
    df["ctr_points"]
)


# Reason code

def get_reason(row):

    if row["stale_points"] >= 2 and row["ctr_points"] == 1:
        return "STALE_AND_WEAK_CTR"

    elif row["stale_points"] >= 2:
        return "STALE_CONTENT"

    elif row["ctr_points"] == 1:
        return "WEAK_CTR_POSITION"

    else:
        return "NO_STRONG_SIGNAL"


df["reason_code"] = df.apply(get_reason, axis=1)


# Action label
def get_action(score):

    if score >= 3:
        return "HIGH_PRIORITY_REFRESH"

    elif score >= 1:
        return "REVIEW_REFRESH"

    else:
        return "LOW_PRIORITY_REVIEW"


df["action"] = df["score"].apply(get_action)


# Rank the pages

ranked_queue = df.sort_values(
    by=["score", "days_since_last_update"],
    ascending=[False, False]
).reset_index(drop=True)

ranked_queue["rank"] = ranked_queue.index + 1


# Select output columns

output_columns = [
    "rank",
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "action",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

baseline_action_score = ranked_queue[output_columns].copy()


# Write required CSV

from pathlib import Path

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

baseline_action_score.to_csv(
    output_path,
    index=False
)

print("Ranked queue created successfully.")
print("Rows:", len(baseline_action_score))
print("Saved to:", output_path)

display(baseline_action_score.head(20))

Ranked queue created successfully.
Rows: 30000
Saved to: work/outputs/baseline_action_score.csv


,rank,content_id,client_id,score,reason_code,action,days_since_last_update,ctr,avg_position
0,1,content_55a5b1c46474,client_4ec9599fc2,3,STALE_AND_WEAK_CTR,HIGH_PRIORITY_REFRESH,373,0.0,7.5
1,2,content_f6fdf87348f6,client_4ec9599fc2,3,STALE_AND_WEAK_CTR,HIGH_PRIORITY_REFRESH,373,0.0,32.5
2,3,content_8d56efff1e71,client_4ec9599fc2,3,STALE_AND_WEAK_CTR,HIGH_PRIORITY_REFRESH,372,0.0,35.0
3,4,content_1b4ec72dafd4,client_4ec9599fc2,3,STALE_AND_WEAK_CTR,HIGH_PRIORITY_REFRESH,372,0.0,7.0
4,5,content_e2b702f4f92b,client_4ec9599fc2,3,STALE_AND_WEAK_CTR,HIGH_PRIORITY_REFRESH,334,0.0,9.3
5,6,content_06e19c6486b0,client_4ec9599fc2,3,STALE_AND_WEAK_CTR,HIGH_PRIORITY_REFRESH,334,0.0,5.0
6,7,content_7a888d3d99c8,client_19581e27de,3,STALE_AND_WEAK_CTR,HIGH_PRIORITY_REFRESH,313,0.0,67.6
7,8,content_6476d1d8c050,client_19581e27de,3,STALE_AND_WEAK_CTR,HIGH_PRIORITY_REFRESH,313,0.0,67.8
8,9,content_94991fe6268c,client_19581e27de,3,STALE_AND_WEAK_CTR,HIGH_PRIORITY_REFRESH,313,0.0,12.4
9,10,content_02b0d6e30129,client_19581e27de,3,STALE_AND_WEAK_CTR,HIGH_PRIORITY_REFRESH,313,0.0,6.9


## Top-20 review

I reviewed the first 20 pages from the ranked queue.

For each page, I record:

- the recommended action;
- the reason code explaining why it was ranked;
- a confidence note;
- what could make the recommendation wrong.

The baseline is a decision-support tool, so a high score does not guarantee that a page should actually be refreshed. Human review is still required.

In [3]:
# Select the top 20 ranked pages
top20 = baseline_action_score.head(20).copy()

# Add a simple confidence note based on the rule score
def confidence_note(score):
    if score >= 3:
        return "High confidence from both refresh signals."
    elif score >= 1:
        return "Moderate confidence; one refresh signal is present."
    else:
        return "Low confidence; weak evidence for refresh."

top20["confidence_note"] = top20["score"].apply(confidence_note)

# Explain what could make each recommendation wrong
def what_would_make_it_wrong(row):

    if row["reason_code"] == "STALE_AND_WEAK_CTR":
        return (
            "The page may already be appropriate despite being old, "
            "or the low CTR may be caused by search intent rather than stale content."
        )

    elif row["reason_code"] == "STALE_CONTENT":
        return (
            "The content may still be accurate and useful even though "
            "it has not been updated recently."
        )

    elif row["reason_code"] == "WEAK_CTR_POSITION":
        return (
            "The CTR may be reasonable for the query intent or SERP features "
            "even though it is below the simple threshold."
        )

    else:
        return (
            "The rule may not capture other important reasons why a page "
            "needs or does not need a refresh."
        )


top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

# Display the required Top-20 review
display(
    top20[
        [
            "rank",
            "content_id",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_55a5b1c46474,HIGH_PRIORITY_REFRESH,STALE_AND_WEAK_CTR,High confidence from both refresh signals.,The page may already be appropriate despite be...
1,2,content_f6fdf87348f6,HIGH_PRIORITY_REFRESH,STALE_AND_WEAK_CTR,High confidence from both refresh signals.,The page may already be appropriate despite be...
2,3,content_8d56efff1e71,HIGH_PRIORITY_REFRESH,STALE_AND_WEAK_CTR,High confidence from both refresh signals.,The page may already be appropriate despite be...
3,4,content_1b4ec72dafd4,HIGH_PRIORITY_REFRESH,STALE_AND_WEAK_CTR,High confidence from both refresh signals.,The page may already be appropriate despite be...
4,5,content_e2b702f4f92b,HIGH_PRIORITY_REFRESH,STALE_AND_WEAK_CTR,High confidence from both refresh signals.,The page may already be appropriate despite be...
5,6,content_06e19c6486b0,HIGH_PRIORITY_REFRESH,STALE_AND_WEAK_CTR,High confidence from both refresh signals.,The page may already be appropriate despite be...
6,7,content_7a888d3d99c8,HIGH_PRIORITY_REFRESH,STALE_AND_WEAK_CTR,High confidence from both refresh signals.,The page may already be appropriate despite be...
7,8,content_6476d1d8c050,HIGH_PRIORITY_REFRESH,STALE_AND_WEAK_CTR,High confidence from both refresh signals.,The page may already be appropriate despite be...
8,9,content_94991fe6268c,HIGH_PRIORITY_REFRESH,STALE_AND_WEAK_CTR,High confidence from both refresh signals.,The page may already be appropriate despite be...
9,10,content_02b0d6e30129,HIGH_PRIORITY_REFRESH,STALE_AND_WEAK_CTR,High confidence from both refresh signals.,The page may already be appropriate despite be...


## Weak picks + leakage check

I also inspect weak picks to see whether the rule produces recommendations that look questionable.

For the leakage check, I make sure that the baseline does not use future-window metrics, labels, or an existing product decision as an input.

The rule uses only:

- `days_since_last_update`
- `ctr`
- `avg_position`

These are the signals used to construct the baseline score.

The baseline should be treated as a transparent ranking rule rather than as a guaranteed prediction.

In [4]:
# Weak picks
weak_picks = baseline_action_score[
    baseline_action_score["action"] == "LOW_PRIORITY_REVIEW"
].head(10)

print("Weak picks:")
display(weak_picks)

# Leakage check
future_or_label_columns = [
    "label",
    "target",
    "future",
    "future_clicks",
    "future_impressions",
    "future_sessions",
    "outcome",
    "health_score",
    "priority_score",
    "action_type"
]

found_leakage_columns = [
    col for col in future_or_label_columns
    if col in df.columns
]

print("\nLeakage check:")

if found_leakage_columns:
    print("Potential leakage columns found:", found_leakage_columns)
else:
    print("No future-window or label-derived columns used by the baseline.")

# Confirm exactly which columns were used by the rule
rule_inputs = [
    "days_since_last_update",
    "ctr",
    "avg_position"
]

print("\nRule input columns:")
print(rule_inputs)

Weak picks:


,rank,content_id,client_id,score,reason_code,action,days_since_last_update,ctr,avg_position
19421,19422,content_0628102f4044,client_434c9b5ae5,0,NO_STRONG_SIGNAL,LOW_PRIORITY_REVIEW,89,0.20,7.7
19422,19423,content_41eb625c4aca,client_434c9b5ae5,0,NO_STRONG_SIGNAL,LOW_PRIORITY_REVIEW,89,0.21,13.2
19423,19424,content_f562786a3f25,client_434c9b5ae5,0,NO_STRONG_SIGNAL,LOW_PRIORITY_REVIEW,89,0.26,7.6
19424,19425,content_d0cadc3e2773,client_434c9b5ae5,0,NO_STRONG_SIGNAL,LOW_PRIORITY_REVIEW,89,0.06,6.7
19425,19426,content_24706e54a64c,client_434c9b5ae5,0,NO_STRONG_SIGNAL,LOW_PRIORITY_REVIEW,89,0.31,7.2
19426,19427,content_e5b7ee456c43,client_434c9b5ae5,0,NO_STRONG_SIGNAL,LOW_PRIORITY_REVIEW,89,0.14,6.9
19427,19428,content_083bac35aa08,client_434c9b5ae5,0,NO_STRONG_SIGNAL,LOW_PRIORITY_REVIEW,89,0.07,13.7
19428,19429,content_696f7cd298e3,client_434c9b5ae5,0,NO_STRONG_SIGNAL,LOW_PRIORITY_REVIEW,89,0.29,7.5
19429,19430,content_31c4d526956d,client_434c9b5ae5,0,NO_STRONG_SIGNAL,LOW_PRIORITY_REVIEW,89,0.61,7.5
19430,19431,content_4ef69f179958,client_434c9b5ae5,0,NO_STRONG_SIGNAL,LOW_PRIORITY_REVIEW,89,0.40,7.7



Leakage check:
No future-window or label-derived columns used by the baseline.

Rule input columns:
['days_since_last_update', 'ctr', 'avg_position']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.